In [1]:
# %matplotlib inline          # line 1
# import matplotlib.pyplot as plt  # line 2

In [2]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pandas as pd
import muon as mu
import scanpy as sc
import scirpy as ir
np.random.seed(42)
import random
random.seed(42)

2026-05-12 17:05:01.975621: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-12 17:05:01.975980: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-12 17:05:02.012641: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-12 17:05:04.021721: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation or

In [3]:
import sys
# sys.path.append(r"E:\Python code\Machine learning\JupyterNote\Bio_CRC\Data processing\functions")
sys.path.append(r"/ihome/ylee/yiz133/Code/Data processing/functions/")
import mdata_utils 

path = (r"/ix1/ylee/Yifan_Zhang/Code_data/external/COMBAT paired T/")
mdata = mu.read(path + "common_HV200.h5mu")

In [4]:
adata_raw = sc.read_h5ad(path + "COMBAT-CITESeq-DATA.h5ad")
adata_raw.obs = adata_raw.obs.rename(columns={'TCR_clone_count': 'clone_id_size', 'TCR_clone_ID':'clone_id'})

In [5]:
mdata['gex'].obs = mdata['gex'].obs.join(
    adata_raw.obs[['clone_id_size']],
    how='left'
)
mdata['gex'].obs = mdata['gex'].obs.join(
    adata_raw.obs[['clone_id']],
    how='left'
)
mdata

MuData object with n_obs × n_vars = 401488 × 2477
  2 modalities
    gex:	401488 x 2477
      obs:	'Annotation_cluster_id', 'Annotation_cluster_name', 'Annotation_minor_subset', 'Annotation_major_subset', 'Annotation_cell_type', 'GEX_region', 'QC_ngenes', 'QC_total_UMI', 'QC_pct_mitochondrial', 'QC_scrub_doublet_scores', 'COMBAT_ID', 'scRNASeq_sample_ID', 'COMBAT_participant_timepoint_ID', 'Source', 'Age', 'Sex', 'Race', 'BMI', 'Hospitalstay', 'Death28', 'Institute', 'PreExistingHeartDisease', 'PreExistingLungDisease', 'PreExistingKidneyDisease', 'PreExistingDiabetes', 'PreExistingHypertension', 'PreExistingImmunocompromised', 'Smoking', 'Symptomatic', 'Requiredvasoactive', 'Respiratorysupport', 'SARSCoV2PCR', 'Outcome', 'TimeSinceOnset', 'Ethnicity', 'Tissue', 'DiseaseClassification', 'Pool_ID', 'Channel_ID', 'clone_id_size', 'clone_id'
      var:	'gene_ids', 'feature_types', 'hvg_union_COMBAT_ID'
      uns:	'Institute', 'ObjectCreateDate', 'Source_colors', 'Technology', 'genome_annotation_version', 'hvg_union_COMBAT_ID_k200', 'hvg_union_COMBAT_ID_meta'
      obsm:	'X_umap', 'X_umap_source'
      layers:	'raw'
    airr:	401488 x 0
      obs:	'receptor_type', 'receptor_subtype', 'chain_pairing'
      uns:	'chain_indices', 'scirpy_version'
      obsm:	'airr', 'chain_indices'

# TCR

In [6]:
ir.pp.index_chains(mdata)
ir.tl.chain_qc(mdata)
mdata.update()
mdata

MuData object with n_obs × n_vars = 401488 × 2477
  2 modalities
    gex:	401488 x 2477
      obs:	'Annotation_cluster_id', 'Annotation_cluster_name', 'Annotation_minor_subset', 'Annotation_major_subset', 'Annotation_cell_type', 'GEX_region', 'QC_ngenes', 'QC_total_UMI', 'QC_pct_mitochondrial', 'QC_scrub_doublet_scores', 'COMBAT_ID', 'scRNASeq_sample_ID', 'COMBAT_participant_timepoint_ID', 'Source', 'Age', 'Sex', 'Race', 'BMI', 'Hospitalstay', 'Death28', 'Institute', 'PreExistingHeartDisease', 'PreExistingLungDisease', 'PreExistingKidneyDisease', 'PreExistingDiabetes', 'PreExistingHypertension', 'PreExistingImmunocompromised', 'Smoking', 'Symptomatic', 'Requiredvasoactive', 'Respiratorysupport', 'SARSCoV2PCR', 'Outcome', 'TimeSinceOnset', 'Ethnicity', 'Tissue', 'DiseaseClassification', 'Pool_ID', 'Channel_ID', 'clone_id_size', 'clone_id'
      var:	'gene_ids', 'feature_types', 'hvg_union_COMBAT_ID'
      uns:	'Institute', 'ObjectCreateDate', 'Source_colors', 'Technology', 'genome_annotation_version', 'hvg_union_COMBAT_ID_k200', 'hvg_union_COMBAT_ID_meta'
      obsm:	'X_umap', 'X_umap_source'
      layers:	'raw'
    airr:	401488 x 0
      obs:	'receptor_type', 'receptor_subtype', 'chain_pairing'
      uns:	'chain_indices', 'scirpy_version'
      obsm:	'airr', 'chain_indices'

In [7]:
mdata['airr'].obs['chain_pairing'].value_counts()

chain_pairing
single pair        302613
orphan VDJ          39446
extra VJ            33882
orphan VJ           14960
extra VDJ            7475
two full chains      3112
Name: count, dtype: int64

In [8]:
# ir.pp.ir_dist(
#     mdata,
#     metric='alignment',
#     sequence="aa",
#     cutoff=5,
# )

# ir.tl.define_clonotypes(mdata, receptor_arms="all", 
#                         dual_ir="primary_only", 
#                         within_group="gex:COMBAT_ID",
#                         same_v_gene=True,
#                         same_j_gene=True)

In [9]:
# fig, ax = plt.subplots(figsize=(10, 5))
# _ = ir.pl.clonal_expansion(
#     mdata, 
#     target_col= "airr:clone_id",
#     groupby= "gex:Annotation_major_subset",  # Use the new combined column
#     breakpoints=(1, 5, 20), 
#     ax = ax
#     #normalize=False
# )

In [10]:
clone_thresh = 2
mdata.obs['cloned'] = mdata['gex'].obs['clone_id_size'] >= clone_thresh
mdata.obs['cloned'].value_counts()


cloned
False    325419
True      76069
Name: count, dtype: int64

# TCR embedings

In [11]:
import TCR_embedings

In [12]:
meta_airr = ir.get.airr(mdata['airr'], ["cdr3_aa", "v_call", "j_call"] ,  ('VJ_1', 'VDJ_1'))
mdata.obs = mdata.obs.join(meta_airr)
mdata.update()
mdata

MuData object with n_obs × n_vars = 401488 × 2477
  obs:	'cloned', 'VJ_1_cdr3_aa', 'VJ_1_v_call', 'VJ_1_j_call', 'VDJ_1_cdr3_aa', 'VDJ_1_v_call', 'VDJ_1_j_call'
  2 modalities
    gex:	401488 x 2477
      obs:	'Annotation_cluster_id', 'Annotation_cluster_name', 'Annotation_minor_subset', 'Annotation_major_subset', 'Annotation_cell_type', 'GEX_region', 'QC_ngenes', 'QC_total_UMI', 'QC_pct_mitochondrial', 'QC_scrub_doublet_scores', 'COMBAT_ID', 'scRNASeq_sample_ID', 'COMBAT_participant_timepoint_ID', 'Source', 'Age', 'Sex', 'Race', 'BMI', 'Hospitalstay', 'Death28', 'Institute', 'PreExistingHeartDisease', 'PreExistingLungDisease', 'PreExistingKidneyDisease', 'PreExistingDiabetes', 'PreExistingHypertension', 'PreExistingImmunocompromised', 'Smoking', 'Symptomatic', 'Requiredvasoactive', 'Respiratorysupport', 'SARSCoV2PCR', 'Outcome', 'TimeSinceOnset', 'Ethnicity', 'Tissue', 'DiseaseClassification', 'Pool_ID', 'Channel_ID', 'clone_id_size', 'clone_id'
      var:	'gene_ids', 'feature_types', 'hvg_union_COMBAT_ID'
      uns:	'Institute', 'ObjectCreateDate', 'Source_colors', 'Technology', 'genome_annotation_version', 'hvg_union_COMBAT_ID_k200', 'hvg_union_COMBAT_ID_meta'
      obsm:	'X_umap', 'X_umap_source'
      layers:	'raw'
    airr:	401488 x 0
      obs:	'receptor_type', 'receptor_subtype', 'chain_pairing'
      uns:	'chain_indices', 'scirpy_version'
      obsm:	'airr', 'chain_indices'

In [13]:
obs = mdata.obs.copy()

n_alpha = obs['VJ_1_cdr3_aa'].nunique()
n_beta  = obs['VDJ_1_cdr3_aa'].nunique()

print(f"Unique alpha-chain (VJ) clonotypes:  {n_alpha}")
print(f"Unique beta-chain  (VDJ) clonotypes: {n_beta}")
print(f"Total cells: {len(obs)}")

Unique alpha-chain (VJ) clonotypes:  196033
Unique beta-chain  (VDJ) clonotypes: 291416
Total cells: 401488


In [14]:
# Atchley factors for the 20 amino acids
atchley_factors = {
    'A': [ 0.591, -1.302, -0.733,  1.570, -0.146],  # Alanine
    'R': [ 1.538,  0.055,  1.502,  0.440,  2.897],  # Arginine
    'N': [ 0.945,  0.828,  1.299, -0.169,  0.933],  # Asparagine
    'D': [ 1.050,  0.302, -3.656, -0.259, -3.242],  # Aspartic acid
    'C': [-1.343,  0.465, -0.862, -1.020, -0.255],  # Cysteine
    'Q': [ 0.931,  0.179, -3.005, -0.503, -1.853],  # Glutamine
    'E': [ 1.357,  0.113, -3.242, -0.339, -2.192],  # Glutamic acid
    'G': [ 0.384,  1.652,  1.330,  1.045,  2.064],  # Glycine
    'H': [ 0.336, -0.417, -1.673, -1.474, -0.078],  # Histidine
    'I': [-1.239, -0.547,  2.131,  0.393,  0.816],  # Isoleucine
    'L': [-1.019, -0.987, -1.505,  1.266, -0.912],  # Leucine
    'K': [ 1.831, -0.561,  0.533, -0.277,  1.648],  # Lysine
    'M': [-0.663, -1.524,  2.219, -1.005,  1.212],  # Methionine
    'F': [-1.006, -0.590,  1.891, -0.397,  0.412],  # Phenylalanine
    'P': [ 0.189,  2.081, -1.628,  0.421, -1.392],  # Proline
    'S': [ 0.228,  1.399, -4.760,  0.670, -2.647],  # Serine
    'T': [ 0.032,  2.213, -1.455,  0.311, -0.259],  # Threonine
    'W': [-0.595,  0.009,  0.672, -2.128, -0.184],  # Tryptophan
    'Y': [ 0.260,  0.830,  3.097, -0.838,  1.512],  # Tyrosine
    'V': [-1.337, -0.279, -0.544,  1.242, -1.262],  # Valine
}

In [15]:
# both chains
tcr_aa_obs = ['VDJ_1_cdr3_aa', 'VJ_1_cdr3_aa']
tcr_cat_features = ['VDJ_1_j_call', 'VDJ_1_v_call', 'VJ_1_j_call', 'VJ_1_v_call']

tcr_num_features = []

In [16]:
# Vectorize each TCR amino acid column and compute composition & length
for aa_col in tcr_aa_obs:
    # 1. Sequence length
    # Only keep cells with both alpha and beta chains
    lengths = TCR_embedings.compute_sequence_lengths(mdata, aa_col)
    lengths_mask = (lengths> 5) & (lengths< 20)
    len_key = f'{aa_col}_length'
    mdata = mdata[lengths_mask]
    mdata.obs[len_key] = lengths[lengths_mask]
    print(f"Stored {aa_col} lengths in mdata.obs['{len_key}']")
    
    # 2. Atchley factor encoding
    encoded = TCR_embedings.vectorize_tcr_column(mdata, aa_col, atchley_factors)
    key_name = f'X_{aa_col}_atchley'
    mdata.obsm[key_name] = encoded
    print(f"Stored {aa_col} Atchley vectors in mdata.obsm['{key_name}'] with shape {encoded.shape}")
    
    # 2b. Adjacent Atchley factor interactions
    # atchley_positions = encoded.shape[1] // 5
    # pairwise_key = f'X_{aa_col}_atchley_pairwise'
    # if atchley_positions > 1:
    #     encoded_reshaped = encoded.reshape(encoded.shape[0], atchley_positions, 5)
    #     pairwise_features = []
    #     for pos in range(atchley_positions - 1):
    #         current = encoded_reshaped[:, pos, :]
    #         nxt = encoded_reshaped[:, pos + 1, :]
    #         outer = (current[:, :, None] * nxt[:, None, :]).reshape(encoded.shape[0], -1)
    #         pairwise_features.append(outer)
    #     pairwise_matrix = np.concatenate(pairwise_features, axis=1)
    # else:
    #     pairwise_matrix = np.zeros((encoded.shape[0], 0))
    # mdata.obsm[pairwise_key] = pairwise_matrix
    # print(f"Stored {aa_col} adjacent Atchley interactions in mdata.obsm['{pairwise_key}'] with shape {pairwise_matrix.shape}")    
    
    # 3. AA composition (percentage of each of 20 AAs)
    aa_comp = TCR_embedings.compute_aa_composition_matrix(mdata, aa_col)
    comp_key = f'X_{aa_col}_composition'
    mdata.obsm[comp_key] = aa_comp
    print(f"Stored {aa_col} AA composition in mdata.obsm['{comp_key}'] with shape {aa_comp.shape}\n")
    

Stored VDJ_1_cdr3_aa lengths in mdata.obs['VDJ_1_cdr3_aa_length']
Stored VDJ_1_cdr3_aa Atchley vectors in mdata.obsm['X_VDJ_1_cdr3_aa_atchley'] with shape (379283, 100)
Stored VDJ_1_cdr3_aa AA composition in mdata.obsm['X_VDJ_1_cdr3_aa_composition'] with shape (379283, 20)

Stored VJ_1_cdr3_aa lengths in mdata.obs['VJ_1_cdr3_aa_length']
Stored VJ_1_cdr3_aa Atchley vectors in mdata.obsm['X_VJ_1_cdr3_aa_atchley'] with shape (320152, 100)
Stored VJ_1_cdr3_aa AA composition in mdata.obsm['X_VJ_1_cdr3_aa_composition'] with shape (320152, 20)



In [17]:
arrs_tcr = []
for key, value in mdata.obsm.items():
    arrs_tcr.append(value)

In [18]:
# Build from the same obsm keys used for concatenation.
# Exclude existing tcr_embs to avoid recursive double counting on reruns.
tcr_obsm_keys = [k for k in mdata.obsm.keys() if k != 'tcr_embs']
arrs_tcr = [mdata.obsm[k] for k in tcr_obsm_keys]
view_tcr = np.concatenate(arrs_tcr, axis=1)

# Build readable feature names for all concatenated TCR columns
view_tcr_names = []
for key in tcr_obsm_keys:
    value = mdata.obsm[key]
    width = value.shape[1] if getattr(value, 'ndim', 1) > 1 else 1
    view_tcr_names.extend([f"{key}_{i}" for i in range(width)])

print(view_tcr.shape)

# Add chain length
for chain in tcr_aa_obs:
    view_tcr = np.concatenate([view_tcr, mdata.obs[chain + '_length'].to_numpy().reshape(-1, 1)],  axis=1)
    view_tcr_names.append(f"{chain}_length")

# Add clone size
# view_tcr = np.concatenate([view_tcr, mdata['airr'].obs['clone_id_size'].to_numpy().reshape(-1, 1)],  axis=1)
# view_tcr_names.append('clone_id_size')

mdata.uns['tcr_embs_feature_names'] = view_tcr_names
    
print(view_tcr.shape)
print(len(view_tcr_names))

(320152, 242)
(320152, 244)
244


In [19]:
# Perform Canonical Correlation Analysis separately on train and test sets
view_gene = mdata['gex'].X.toarray()

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
view_tcr = scaler.fit_transform(view_tcr)
view_gene = scaler.fit_transform(view_gene)

In [20]:
mdata.obsm['tcr_embs'] = view_tcr

In [21]:
mdata_emb = mdata.copy()
mdata = mdata_emb.copy()

In [25]:
mdata_emb.write_h5mu(path + 'common_HV200_atchEmb.h5mu')